# Assignment 14 — Fine-tune ViT with LoRA (Kaggle GPU)
Dataset: `AI-Lab-Makerere/beans` (3 classes: angular_leaf_spot, bean_rust, healthy).
Base model: `google/vit-base-patch16-224-in21k`. Only LoRA adapters on attention
`query`/`value` + the classifier head are trained. The best checkpoint (by val accuracy)
is saved as a LoRA adapter, then merged into the base model for deployment.

In [ ]:
# Kaggle's preinstalled torchao is too old for recent peft (and unused here)
!pip uninstall -y -q torchao
!pip install -q peft evaluate

In [ ]:
import numpy as np, torch, evaluate
from datasets import load_dataset
from transformers import (AutoImageProcessor, ViTForImageClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model
from torchvision.transforms import (Compose, Normalize, RandomResizedCrop, RandomHorizontalFlip,
                                    Resize, CenterCrop, ToTensor)

BASE = "google/vit-base-patch16-224-in21k"
OUT = "/kaggle/working"
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
ds = load_dataset("AI-Lab-Makerere/beans")
labels = ds["train"].features["labels"].names
id2label = dict(enumerate(labels)); label2id = {v: k for k, v in id2label.items()}
print(ds, labels)

In [ ]:
processor = AutoImageProcessor.from_pretrained(BASE)
size = processor.size["height"]
norm = Normalize(mean=processor.image_mean, std=processor.image_std)
train_tf = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), norm])
eval_tf = Compose([Resize(size), CenterCrop(size), ToTensor(), norm])

def with_tf(tf):
    def f(batch):
        batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["image"]]
        return batch
    return f

ds["train"].set_transform(with_tf(train_tf))
ds["validation"].set_transform(with_tf(eval_tf))
ds["test"].set_transform(with_tf(eval_tf))

def collate(examples):
    return {"pixel_values": torch.stack([e["pixel_values"] for e in examples]),
            "labels": torch.tensor([e["labels"] for e in examples])}

In [ ]:
model = ViTForImageClassification.from_pretrained(BASE, num_labels=len(labels),
                                                  id2label=id2label, label2id=label2id)
lora_cfg = LoraConfig(r=16, lora_alpha=16, lora_dropout=0.1, bias="none",
                      target_modules=["query", "value"], modules_to_save=["classifier"])
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
accuracy = evaluate.load("accuracy")
def compute_metrics(p):
    return accuracy.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

args = TrainingArguments(
    output_dir=f"{OUT}/checkpoints",
    remove_unused_columns=False,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    learning_rate=5e-3, per_device_train_batch_size=32, per_device_eval_batch_size=32,
    num_train_epochs=10, warmup_ratio=0.1, weight_decay=0.01,
    fp16=torch.cuda.is_available(), logging_steps=10,
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    label_names=["labels"], report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
                  data_collator=collate, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer.train()

In [ ]:
print("val :", trainer.evaluate(ds["validation"]))
print("test:", trainer.evaluate(ds["test"]))

Save the best LoRA adapter (small), then merge it into the base weights (full fp32 model
used as input for quantization).

In [ ]:
model.save_pretrained(f"{OUT}/vit-beans-lora")          # best adapter (load_best_model_at_end)
processor.save_pretrained(f"{OUT}/vit-beans-lora")

merged = model.merge_and_unload()
merged.save_pretrained(f"{OUT}/vit-beans-merged")
processor.save_pretrained(f"{OUT}/vit-beans-merged")

In [ ]:
import shutil; shutil.rmtree(f"{OUT}/checkpoints", ignore_errors=True)  # keep output small
!du -sh {OUT}/*